## 1. Dependency

In [1]:
## create high-level ground truth

In [2]:
import csv
import re
import pandas as pd

## 2. Konfigurasi

In [ ]:
dataset = "1sample"

CSV_INPUT = f"results/{dataset}/result-3-low-level-ground-truth.csv"
CSV_OUTPUT_ALL = f"results/{dataset}/result-6-1-high-level-ground-truth.csv"
CSV_OUTPUT_UNIQUE = f"results/{dataset}/result-6-1-high-level-ground-truth-unique-datetime.csv"

In [4]:
def count_lines(path):
    # fast line count
    cnt = 0
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        for _ in f:
            cnt += 1
    return cnt

## 3. Export ke CSV - Versi 1: Semua GT (dengan filter duplicate berurut)

In [5]:
# Tulis ke CSV - Versi 1: Semua GT yang sesuai filter
total_lines = count_lines(CSV_INPUT)
print(f"Total lines in {CSV_INPUT}: {total_lines}")
print("\n" + "="*80)
print("OUTPUT 1: All Ground Truth Events (dengan filter duplicate berurut)")
print("="*80)

processed = 0
written_all = 0
skipped_benign = 0
skipped_duplicate = 0
last_ground_truth_label = None  # Track label sebelumnya untuk deteksi duplikat berurutan

with open(CSV_INPUT, newline='', encoding="utf-8", errors='replace') as f, \
     open(CSV_OUTPUT_ALL, "w", newline='', encoding="utf-8") as out:
    
    reader = csv.DictReader(f)
    writer = csv.writer(out)
    
    # Header
    writer.writerow(['event_id', 'datetime', 'display_name', 'decoded', 'ground_truth_label'])
    
    for row in reader:
        processed += 1
        
        # Ambil nilai kolom
        event_id = row.get('event_id', '')
        datetime_val = row.get('datetime', '')
        display_name = row.get('display_name', '')
        decoded = row.get('decoded', '')
        ground_truth_label = row.get('ground_truth_label', '')
        
        # 1. Skip jika ground_truth_label adalah 'benign' (tidak me-reset last_ground_truth_label)
        if ground_truth_label.lower() == 'benign':
            skipped_benign += 1
            continue
        
        # 2. Skip jika ground_truth_label sama dengan sebelumnya (duplikat berurutan)
        if ground_truth_label == last_ground_truth_label:
            skipped_duplicate += 1
            continue
        
        # Tulis row yang lolos filter
        writer.writerow([event_id, datetime_val, display_name, decoded, ground_truth_label])
        written_all += 1
        
        # Update last label
        last_ground_truth_label = ground_truth_label
        
        # Progress tiap 50.000 baris
        if processed % 50000 == 0:
            print(f"Processed {processed}/{total_lines} lines ({processed/total_lines:.2%})")

print(f"\n=== Output 1 Summary ===")
print(f"Total processed: {processed} lines")
print(f"Skipped (benign): {skipped_benign} lines")
print(f"Skipped (duplicate label): {skipped_duplicate} lines")
print(f"Written to {CSV_OUTPUT_ALL}: {written_all} lines")

Total lines in results/organization-x/result-3-low-level-ground-truth.csv: 216761

OUTPUT 1: All Ground Truth Events (dengan filter duplicate berurut)

=== Output 1 Summary ===
Total processed: 216754 lines
Skipped (benign): 216082 lines
Skipped (duplicate label): 336 lines
Written to results/organization-x/result-6-1-high-level-ground-truth.csv: 336 lines


## 4. Export ke CSV - Versi 2: Unique Datetime

In [6]:
# Tulis ke CSV - Versi 2: Unique datetime (ambil event pertama per datetime)
# INPUT: Output 1 (bukan langsung dari result-3)
total_lines_v2 = count_lines(CSV_OUTPUT_ALL)
print("\n" + "="*80)
print("OUTPUT 2: Unique Datetime (ambil event pertama per datetime)")
print("="*80)
print(f"Input from Output 1: {total_lines_v2} lines")

processed_v2 = 0
written_unique = 0
skipped_duplicate_datetime = 0
seen_datetimes = set()  # Track datetime yang sudah ditulis

with open(CSV_OUTPUT_ALL, newline='', encoding="utf-8", errors='replace') as f, \
     open(CSV_OUTPUT_UNIQUE, "w", newline='', encoding="utf-8") as out:
    
    reader = csv.DictReader(f)
    writer = csv.writer(out)
    
    # Header
    writer.writerow(['event_id', 'datetime', 'display_name', 'decoded', 'ground_truth_label'])
    
    for row in reader:
        processed_v2 += 1
        
        # Ambil nilai kolom
        event_id = row.get('event_id', '')
        datetime_val = row.get('datetime', '')
        display_name = row.get('display_name', '')
        decoded = row.get('decoded', '')
        ground_truth_label = row.get('ground_truth_label', '')
        
        # Skip jika datetime sudah pernah ditulis
        if datetime_val in seen_datetimes:
            skipped_duplicate_datetime += 1
            continue
        
        # Tulis row yang lolos filter
        writer.writerow([event_id, datetime_val, display_name, decoded, ground_truth_label])
        written_unique += 1
        
        # Update tracking
        seen_datetimes.add(datetime_val)
        
        # Progress tiap 50.000 baris
        if processed_v2 % 50000 == 0:
            print(f"Processed {processed_v2}/{total_lines_v2} lines ({processed_v2/total_lines_v2:.2%})")

print(f"\n=== Output 2 Summary ===")
print(f"Total processed: {processed_v2} lines")
print(f"Skipped (duplicate datetime): {skipped_duplicate_datetime} lines")
print(f"Written to {CSV_OUTPUT_UNIQUE}: {written_unique} lines")


OUTPUT 2: Unique Datetime (ambil event pertama per datetime)
Input from Output 1: 337 lines

=== Output 2 Summary ===
Total processed: 336 lines
Skipped (duplicate datetime): 64 lines
Written to results/organization-x/result-6-1-high-level-ground-truth-unique-datetime.csv: 272 lines


## 5. Summary Comparison

In [7]:
print("\n" + "="*80)
print("FINAL SUMMARY - COMPARISON")
print("="*80)
print(f"Output 1 (All GT):                      {written_all} events")
print(f"Output 2 (Unique Datetime):             {written_unique} events")
print("="*80)


FINAL SUMMARY - COMPARISON
Output 1 (All GT):                      336 events
Output 2 (Unique Datetime):             272 events
